# NS-3 TCP Assignment 6 Analysis Notebook

Put this notebook in `assignment_6/` next to the `output/` folder. Run all cells after finishing the NS-3 simulations.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

OUTPUT = Path("output")

def read_cwnd(path):
    df = pd.read_csv(path, sep=r"\s+", names=["time", "old_cwnd", "cwnd"], engine="python")
    df["cwnd_kb"] = df["cwnd"] / 1024.0
    return df

def read_drops(path):
    if not Path(path).exists() or Path(path).stat().st_size == 0:
        return pd.DataFrame({"time": []})
    return pd.read_csv(path, sep=r"\s+", names=["time"], engine="python")

def add_drop_markers(ax, drops, y):
    for t in drops["time"].values:
        ax.scatter([t], [y], marker="x")


## Task 6.1.4 Plot 1: fixed delay = 10 ms
Compare NewReno and Vegas under different bit error rates. Packet drops are marked with x symbols.

In [ ]:
single_runs_delay10 = [
    ("NewReno, BER=1e-5", "614_newreno_ber1e-5_delay10ms"),
    ("NewReno, BER=1e-6", "614_newreno_ber1e-6_delay10ms"),
    ("Vegas, BER=1e-5", "614_vegas_ber1e-5_delay10ms"),
    ("Vegas, BER=1e-6", "614_vegas_ber1e-6_delay10ms"),
]

fig, ax = plt.subplots(figsize=(12, 6))
for label, name in single_runs_delay10:
    cwnd_file = OUTPUT / f"{name}.cwnd"
    drop_file = OUTPUT / f"{name}.rxdrop"
    if cwnd_file.exists():
        df = read_cwnd(cwnd_file)
        ax.plot(df["time"], df["cwnd_kb"], label=label)
        drops = read_drops(drop_file)
        if not drops.empty:
            y = df["cwnd_kb"].max() * 1.03
            add_drop_markers(ax, drops, y)
ax.set_xlabel("Time [s]")
ax.set_ylabel("Congestion window [KB]")
ax.set_title("Task 6.1.4: delay fixed at 10 ms")
ax.grid(True)
ax.legend()
plt.show()

## Task 6.1.4 Plot 2: fixed BER = 1e-5
Compare NewReno and Vegas under different link delays.

In [ ]:
single_runs_ber1e5 = [
    ("NewReno, delay=10ms", "614_newreno_ber1e-5_delay10ms"),
    ("NewReno, delay=20ms", "614_newreno_ber1e-5_delay20ms"),
    ("Vegas, delay=10ms", "614_vegas_ber1e-5_delay10ms"),
    ("Vegas, delay=20ms", "614_vegas_ber1e-5_delay20ms"),
]

fig, ax = plt.subplots(figsize=(12, 6))
for label, name in single_runs_ber1e5:
    cwnd_file = OUTPUT / f"{name}.cwnd"
    drop_file = OUTPUT / f"{name}.rxdrop"
    if cwnd_file.exists():
        df = read_cwnd(cwnd_file)
        ax.plot(df["time"], df["cwnd_kb"], label=label)
        drops = read_drops(drop_file)
        if not drops.empty:
            y = df["cwnd_kb"].max() * 1.03
            add_drop_markers(ax, drops, y)
ax.set_xlabel("Time [s]")
ax.set_ylabel("Congestion window [KB]")
ax.set_title("Task 6.1.4: BER fixed at 1e-5")
ax.grid(True)
ax.legend()
plt.show()

## Task 6.1.5 Suggested analysis text
NewReno increases cWnd aggressively in slow start and then more linearly in congestion avoidance. When a packet drop occurs, the congestion window is reduced. A higher bit error rate produces more non-congestion losses, therefore NewReno may reduce cWnd even when the link is not truly congested. A larger delay increases RTT, so the cWnd grows more slowly over wall-clock time. Vegas reacts to delay/RTT changes earlier and is usually smoother and more conservative than NewReno, because it tries to detect incipient congestion before losses occur.

## Task 6.2.3 / 6.2.5 / 6.2.6: multiple-flow cWnd plots

In [ ]:
def plot_multi_cwnd(expname, title):
    fig, ax = plt.subplots(figsize=(12, 6))
    for flow in [1, 2]:
        f = OUTPUT / f"multi_tcp_{expname}_flow{flow}.cwnd"
        if f.exists():
            df = read_cwnd(f)
            ax.plot(df["time"], df["cwnd_kb"], label=f"flow {flow}")
    ax.set_xlabel("Time [s]")
    ax.set_ylabel("Congestion window [KB]")
    ax.set_title(title)
    ax.grid(True)
    ax.legend()
    plt.show()

for exp, title in [
    ("newreno_base", "Task 6.2.3 NewReno: two TCP flows"),
    ("bic_base", "Task 6.2.3 BIC: two TCP flows"),
    ("newreno_delaychange", "Task 6.2.5 NewReno: receiver-1 delay changes at 20s"),
    ("bic_delaychange", "Task 6.2.5 BIC: receiver-1 delay changes at 20s"),
    ("newreno_udp", "Task 6.2.6 NewReno with UDP starting at 30s"),
    ("bic_udp", "Task 6.2.6 BIC with UDP starting at 30s"),
]:
    plot_multi_cwnd(exp, title)

## Bonus 6.2.4: throughput over time

In [ ]:
def read_throughput(path):
    return pd.read_csv(path, sep="\t")

def plot_throughput(expname):
    f = OUTPUT / f"multi_tcp_{expname}.throughput"
    if not f.exists():
        print(f"Missing {f}")
        return
    df = read_throughput(f)
    # TCP protocol is 6; UDP is 17. TCP ports are 8080 and 8081 in the completed C++ code.
    tcp = df[df["protocol"] == 6]
    fig, ax = plt.subplots(figsize=(12, 6))
    for flow_id, g in tcp.groupby("flowId"):
        ax.plot(g["time"], g["throughputMbps"], label=f"FlowMonitor flow {flow_id}")
    ax.set_xlabel("Time [s]")
    ax.set_ylabel("Throughput [Mbps]")
    ax.set_title(f"Throughput over time: {expname}")
    ax.grid(True)
    ax.legend()
    plt.show()

for exp in ["newreno_base", "bic_base", "newreno_delaychange", "bic_delaychange", "newreno_udp", "bic_udp"]:
    plot_throughput(exp)

## Task 6.2 analysis template
When flow 2 starts at 11 s, the bottleneck link must be shared by two TCP connections. Flow 1 normally reduces its cWnd after congestion/loss signals, and both flows move toward a fairer share of the 10 Mbps bottleneck. With equal RTTs, NewReno should show AIMD-like oscillations around the fair share. BIC can grow more aggressively around large windows and may show different convergence dynamics. When the delay of the Receiver-1 access link is increased at 20 s, flow 1 has a larger RTT and therefore can become disadvantaged, especially for loss-based TCP where throughput is RTT-sensitive. When a 10 Mbps UDP flow starts at 30 s, it occupies a large part of the 10 Mbps bottleneck and does not back off like TCP, so the TCP cWnd values should drop and throughput should decrease.